In [35]:
import ee
import geemap
from typing import Any
import geopandas as gpd
from dotenv import load_dotenv
load_dotenv()

False

## . Configurations

In [36]:
# Folder for out_google_folderputs
exp_folder = "NGA_Abductions"

# Default map centre coordinates
map_centre_coord = (11.10, 11.38)

In [37]:
# Initialize the library
gee_project_name = "" #os.getenv("GEE_PROJECT_NAME")

# Trigger the authentication flow 
try:
    ee.Initialize(project= gee_project_name)
except:
    ee.Authenticate()
    ee.Initialize(project= gee_project_name)

EEException: Cannot authenticate: Invalid request.

## . Helper Functions

In [ ]:
def feature_map(
    feature: ee.FeatureCollection,
    vis_params: dict[str, Any],
    name: str,
    centre_coord: list[float],
    zoom: int = 8,
) -> geemap.Map:
    """Render an Earth Engine FeatureCollection on a satellite basemap."""

    Map = geemap.Map(center=centre_coord, zoom=zoom)
    Map.add_basemap("SATELLITE")
    styled_layer = feature.style(**vis_params)
    Map.add_layer(styled_layer, {}, name)
    return Map


def geodata_map(
    gdf: gpd.GeoDataFrame,
    style: dict[str, Any],
    name: str,
    centre_coord: list[float],
    zoom: int = 8,
) -> geemap.Map:
    """Render a GeoDataFrame as a styled vector layer on a satellite basemap."""

    Map = geemap.Map(center=centre_coord, zoom=zoom)
    Map.add_basemap("SATELLITE")
    Map.add_gdf(gdf, layer_name=name, style=style)
    return Map

In [ ]:
def export_fc_to_drive(
    feature_collection: ee.FeatureCollection,
    export_desc: str,
    export_folder: str = "Land_Use_Conflicts_Nigeria",
    file_format: str = "SHP",
) -> ee.batch.Task:
    """Export an Earth Engine FeatureCollection to Google Drive.

    Args:
        feature_collection: ee.FeatureCollection to export.
        export_desc: Export task description and output filename.
        export_folder: Google Drive destination folder.
        file_format: Export file format (default is "SHP").

    Returns:
        The initiated ee.batch.Task.
    """

    task = ee.batch.Export.table.toDrive(
        collection=feature_collection,
        description=export_desc,
        folder=export_folder,
        fileFormat=file_format,
    )

    task.start()

    return task

## . Boundary Data Download & Export

In [ ]:
# FAO Global Administrative Unit Layers (GAUL) – 2024 Edition
gaul_2024_l1 = ee.FeatureCollection("projects/sat-io/open-datasets/FAO/GAUL/GAUL_2024_L1")
gaul_2024_l2 = ee.FeatureCollection("projects/sat-io/open-datasets/FAO/GAUL/GAUL_2024_L2")
#print(gaul_2024_l2.limit(0).getInfo()["columns"])


nga_bdry_l0 = (gaul_2024_l1
                    .filter(ee.Filter.eq("gaul0_name", "Nigeria"))
                    .union()
                )

nga_bdry_l1 = gaul_2024_l1.filter(ee.Filter.eq("gaul0_name", "Nigeria"))
nga_bdry_l2 = gaul_2024_l2.filter(ee.Filter.eq("gaul0_name", "Nigeria"))


print("Nigerian States: \n")
print(nga_bdry_l1.aggregate_array("gaul1_name").sort().getInfo())

Nigerian States: 

['Abia', 'Adamawa', 'Akwa Lbom', 'Anambra', 'Bauchi', 'Bayelsa', 'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti', 'Enugu', 'Federal Capital Territory', 'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nassarawa', 'Niger', 'Ogun', 'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto', 'Taraba', 'Yobe', 'Zamfara']


In [ ]:
# Visualize Nigerian administrative boundary
vis_params_bdry_1 = {"color": "red", "fillColor": "00000000", "width": 2}
vis_params_bdry_2 = {"color": "blue", "fillColor": "00000000", "width": 2}


roi_map = feature_map(nga_bdry_l0, vis_params_bdry_1, "Adm0 - Nigeria", map_centre_coord)
roi_map.addLayer( nga_bdry_l1.style(**vis_params_bdry_2), {}, "Adm1- Nigeria")
roi_map

Map(center=[11.1, 11.38], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [ ]:
# Export boundaries
""" 
# ADM0
task_l0 = export_fc_to_drive(
    feature_collection=nga_bdry_l0,
    export_desc="NGA_GAUL2024_ADM0",
    export_folder=exp_folder,
)
"""

# ADM1
task_l1 = export_fc_to_drive(
    feature_collection=nga_bdry_l1,
    export_desc="NGA_GAUL2024_ADM1",
    export_folder=exp_folder,
    file_format="GeoJSON"
)

# ADM2
task_l2 = export_fc_to_drive(
    feature_collection=nga_bdry_l2,
    export_desc="NGA_GAUL2024_ADM2",
    export_folder=exp_folder,
    file_format="GeoJSON"
)

RefreshError: ('invalid_grant: Token has been expired or revoked.', {'error': 'invalid_grant', 'error_description': 'Token has been expired or revoked.'})